# Object Detection (PyTorch)

This lab is similar to the previous lab, except now instead of printing out the bounding box coordinates, you can visualize these bounding boxes on top of the image!

> This notebook is a PyTorch port of the original TensorFlow Hub lab. The pre-trained detectors come from `torchvision.models.detection` instead of TF Hub. They were trained on the [COCO](https://cocodataset.org/) dataset (80 classes) rather than Open Images, so the class names you will see are COCO's.

## Setup

In [ ]:
# For running inference with the torchvision detection models.
import torch
import torchvision
from torchvision.models import detection
from torchvision.transforms.functional import convert_image_dtype

# For downloading the image.
import matplotlib.pyplot as plt
import tempfile
from urllib.request import urlopen, Request
from io import BytesIO

# For drawing onto the image.
import numpy as np
from PIL import Image
from PIL import ImageColor
from PIL import ImageDraw
from PIL import ImageFont
from PIL import ImageOps

# For measuring the inference time.
import time

# Check available accelerator devices.
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Running on device: %s" % device)

### Select and load the model
As in the previous lab, you can choose an object detection model. Here are two that we've selected for you:
* [SSDLite + MobileNet V3](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html) small and fast.
* [Faster R-CNN + ResNet50 FPN v2](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html): high accuracy

In [ ]:
# you can switch the commented lines here to pick the other model

# ssdlite + mobilenet v3
model_handle = "ssdlite320_mobilenet_v3_large"

# You can choose faster r-cnn + resnet50 instead
#model_handle = "fasterrcnn_resnet50_fpn_v2"

#### Load the model

Next, you'll load the model specified by the `model_handle`.
- torchvision downloads the pre-trained COCO weights the first time a model is used.

In [ ]:
model_builders = {
    "ssdlite320_mobilenet_v3_large": (detection.ssdlite320_mobilenet_v3_large, detection.SSDLite320_MobileNet_V3_Large_Weights.COCO_V1),
    "fasterrcnn_resnet50_fpn_v2": (detection.fasterrcnn_resnet50_fpn_v2, detection.FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1),
}

builder, weights = model_builders[model_handle]
model = builder(weights=weights).to(device)

# the COCO class names, indexed by the label ids the model outputs
class_names = weights.meta["categories"]

#### Put the model in inference mode

A TF Hub model exposes *signatures*; a torchvision detection model instead has two modes. In training mode it returns losses, in evaluation mode (`model.eval()`) it returns the detections: a list with one dictionary per image holding the `boxes` (in pixels, `[xmin, ymin, xmax, ymax]`), `labels` and `scores`.

In [ ]:
# take a look at the model's documentation
print(model.__class__.__name__)
print((model.__class__.__doc__ or "")[:900])

Please put the model in evaluation mode to use it as an object detector.

In [ ]:
detector = model.eval()

### download_and_resize_image

As you saw in the previous lab, this function downloads an image specified by a given "url", pre-processes it, and then saves it to disk.
- What new compared to the previous lab is that you an display the image if you set the parameter `display=True`.

In [ ]:
def display_image(image):
    '''
    Displays an image inside the notebook. This is used by download_and_resize_image().

    Args:
      image -- a PIL image or a (height, width, 3) array
    '''
    fig = plt.figure(figsize=(20, 15))
    plt.grid(False)
    plt.imshow(image)

def font_getsize(font, text):
    '''
    Get text width and height for a given font.
    This is used by draw_bounding_box_on_image().

    Args:
      font (ImageFont) -- the font the text will be drawn in
      text (string) -- the text to measure

    Returns:
      (int, int) -- the width and height of the rendered text
    '''

    left, top, right, bottom = font.getbbox(text)

    return right - left, bottom - top


def download_and_resize_image(url, new_width=256, new_height=256, display=False):
    '''
    Fetches an image online, resizes it and saves it locally.

    Args:
        url (string) -- link to the image
        new_width (int) -- size in pixels used for resizing the width of the image
        new_height (int) -- size in pixels used for resizing the length of the image

    Returns:
        (string) -- path to the saved image
    '''


    # create a temporary file ending with ".jpg"
    _, filename = tempfile.mkstemp(suffix=".jpg")

    # opens the given URL (a User-Agent header is needed, otherwise Wikimedia refuses the request)
    response = urlopen(Request(url, headers={'User-Agent': 'Mozilla/5.0'}))

    # reads the image fetched from the URL
    image_data = response.read()

    # puts the image data in memory buffer
    image_data = BytesIO(image_data)

    # opens the image
    pil_image = Image.open(image_data)

    # resizes the image. will crop if aspect ratio is different.
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), Image.Resampling.LANCZOS)

    # converts to the RGB colorspace
    pil_image_rgb = pil_image.convert("RGB")

    # saves the image to the temporary file created earlier
    pil_image_rgb.save(filename, format="JPEG", quality=90)

    print("Image downloaded to %s." % filename)

    if display:
        display_image(pil_image)


    return filename

### Select and load an image
Load a public image from Open Images v4, save locally, and display.

In [ ]:
# By Heiko Gorski, Source: https://commons.wikimedia.org/wiki/File:Naxos_Taverna.jpg
image_url = "https://upload.wikimedia.org/wikipedia/commons/6/60/Naxos_Taverna.jpg"  #@param
downloaded_image_path = download_and_resize_image(image_url, 1280, 856, True)

### Draw bounding boxes

To build on what you saw in the previous lab, you can now visualize the predicted bounding boxes, overlaid on top of the image.  
- You can use `draw_boxes` to do this.  It will use `draw_bounding_box_on_image` to draw the bounding boxes.

In [ ]:
def draw_bounding_box_on_image(image,
                               ymin,
                               xmin,
                               ymax,
                               xmax,
                               color,
                               font,
                               thickness=4,
                               display_str_list=()):

    """
    Adds a bounding box to an image.

    Args:
        image -- the image object
        ymin -- bounding box coordinate
        xmin -- bounding box coordinate
        ymax -- bounding box coordinate
        xmax -- bounding box coordinate
        color -- color for the bounding box edges
        font -- font for class label
        thickness -- edge thickness of the bounding box
        display_str_list -- class labels for each object detected


    Returns:
        No return.  The function modifies the `image` argument
                    that gets passed into this function

    """
    draw = ImageDraw.Draw(image)
    im_width, im_height = image.size

    # scale the bounding box coordinates to the height and width of the image
    (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                ymin * im_height, ymax * im_height)

    # define the four edges of the detection box
    draw.line([(left, top), (left, bottom), (right, bottom), (right, top),
             (left, top)],
            width=thickness,
            fill=color)

    # If the total height of the display strings added to the top of the bounding
    # box exceeds the top of the image, stack the strings below the bounding box
    # instead of above.
    display_str_heights = [font_getsize(font,ds)[1] for ds in display_str_list]
    # Each display_str has a top and bottom margin of 0.05x.
    total_display_str_height = (1 + 2 * 0.05) * sum(display_str_heights)

    if top > total_display_str_height:
        text_bottom = top
    else:
        text_bottom = top + total_display_str_height

    # Reverse list and print from bottom to top.
    for display_str in display_str_list[::-1]:
        text_width, text_height = font_getsize(font, display_str)
        margin = np.ceil(0.05 * text_height)
        draw.rectangle([(left, text_bottom - text_height - 2 * margin),
                        (left + text_width, text_bottom)],
                       fill=color)
        draw.text((left + margin, text_bottom - text_height - margin),
                  display_str,
                  fill="black",
                  font=font)
        text_bottom -= text_height - 2 * margin


def draw_boxes(image, boxes, class_names, scores, max_boxes=10, min_score=0.1):
    """
    Overlay labeled boxes on an image with formatted scores and label names.

    Args:
        image -- the image as a numpy array
        boxes -- list of detection boxes in normalized [ymin, xmin, ymax, xmax] format
        class_names -- list of classes for each detected object
        scores -- numbers showing the model's confidence in detecting that object
        max_boxes -- maximum detection boxes to overlay on the image (default is 10)
        min_score -- minimum score required to display a bounding box

    Returns:
        image -- the image after detection boxes and classes are overlaid on the original image.
    """
    colors = list(ImageColor.colormap.values())

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf",
                              25)
    except IOError:
        print("Font not found, using default font.")
        font = ImageFont.load_default(size=25)

    for i in range(min(boxes.shape[0], max_boxes)):

        # only display detection boxes that have the minimum score or higher
        if scores[i] >= min_score:
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            display_str = "{}: {}%".format(class_names[i],
                                         int(100 * scores[i]))
            color = colors[hash(class_names[i]) % len(colors)]
            image_pil = Image.fromarray(np.uint8(image)).convert("RGB")

            # draw one bounding box and overlay the class labels onto the image
            draw_bounding_box_on_image(image_pil,
                                       ymin,
                                       xmin,
                                       ymax,
                                       xmax,
                                       color,
                                       font,
                                       display_str_list=[display_str])
            np.copyto(image, np.array(image_pil))

    return image

### run_detector

This function will take in the object detection model `detector` and the path to a sample image, then use this model to detect objects.
- This time, run_detector also calls `draw_boxes` to draw the predicted bounding boxes.
- torchvision returns the boxes in pixels as `[xmin, ymin, xmax, ymax]`; they are converted to the normalized `[ymin, xmin, ymax, xmax]` format that `draw_boxes` expects (this is the format TF Hub models returned).

In [ ]:
def load_img(path):
    '''
    Loads a JPEG image and converts it to a tensor.

    Args:
        path (string) -- path to a locally saved JPEG image

    Returns:
        (tensor) -- an image tensor of shape (3, height, width) with uint8 values
    '''

    # read and decode the file
    img = torchvision.io.read_image(path, mode=torchvision.io.ImageReadMode.RGB)

    return img


def run_detector(detector, path, class_names, device):
    '''
    Runs inference on a local file using an object detection model.

    Args:
        detector (model) -- an object detection model from torchvision
        path (string) -- path to an image saved locally
        class_names (list of str) -- COCO category names indexed by label id
        device (torch.device) -- device the model runs on
    '''

    # load an image tensor from a local file path
    img = load_img(path)

    # convert to float in the range [0, 1] and move to the device
    # (torchvision detectors take a list of image tensors instead of a batch with a leading dimension)
    converted_img = convert_image_dtype(img, torch.float32).to(device)

    # run inference using the model
    start_time = time.time()
    with torch.no_grad():
        result = detector([converted_img])[0]
    end_time = time.time()

    # save the results in a dictionary
    result = {key:value.cpu().numpy() for key,value in result.items()}

    # convert the boxes from pixel [xmin, ymin, xmax, ymax] to normalized [ymin, xmin, ymax, xmax]
    height, width = img.shape[1], img.shape[2]
    xmin, ymin, xmax, ymax = result["boxes"].T
    result["detection_boxes"] = np.stack([ymin / height, xmin / width, ymax / height, xmax / width], axis=1)
    result["detection_scores"] = result["scores"]
    result["detection_class_entities"] = [class_names[label] for label in result["labels"]]

    # print results
    print("Found %d objects." % len(result["detection_scores"]))
    print("Inference time: ", end_time-start_time)

    # draw predicted boxes over the image (as a (height, width, 3) numpy array)
    image_with_boxes = draw_boxes(
      img.permute(1, 2, 0).numpy(), result["detection_boxes"],
      result["detection_class_entities"], result["detection_scores"])

    # display the image
    display_image(image_with_boxes)

### Run the detector on your selected image!

In [ ]:
run_detector(detector, downloaded_image_path, class_names, device)

### Run the detector on more images
Perform inference on some additional images of your choice and check how long inference takes.

In [ ]:
image_urls = [
  # Source: https://commons.wikimedia.org/wiki/File:The_Coleoptera_of_the_British_islands_(Plate_125)_(8592917784).jpg
  "https://upload.wikimedia.org/wikipedia/commons/1/1b/The_Coleoptera_of_the_British_islands_%28Plate_125%29_%288592917784%29.jpg",
  # By Américo Toledano, Source: https://commons.wikimedia.org/wiki/File:Biblioteca_Maim%C3%B3nides,_Campus_Universitario_de_Rabanales_007.jpg
  "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg/960px-Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg",
  # Source: https://commons.wikimedia.org/wiki/File:The_smaller_British_birds_(8053836633).jpg
  "https://upload.wikimedia.org/wikipedia/commons/0/09/The_smaller_British_birds_%288053836633%29.jpg",
  ]

def detect_img(image_url, detector, class_names, device):
    '''
    Downloads an image, runs detection on it, and reports the total time.

    Args:
      image_url (string) -- link to the image to fetch
      detector (nn.Module) -- torchvision detection model in eval mode
      class_names (list of str) -- COCO category names indexed by label id
      device (torch.device) -- device the model runs on
    '''
    start_time = time.time()
    image_path = download_and_resize_image(image_url, 640, 480)
    run_detector(detector, image_path, class_names, device)
    end_time = time.time()
    print("Inference time:",end_time-start_time)

In [ ]:
detect_img(image_urls[0], detector, class_names, device)

In [ ]:
detect_img(image_urls[1], detector, class_names, device)

In [ ]:
detect_img(image_urls[2], detector, class_names, device)